In [ ]:
##START BASE HCNN ONLY MODEL

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, TensorBoard


# 1. CONFIGURATION

image_dir = "/content/sample_data"   # CHANGE if needed
img_height = 224
img_width = 224
batch_size = 32
EPOCHS = 30

class_names = [
    "GG0038ON_011",
    "GG0062S_003",
    "GG0131O_001",
    "SL 124_001",
    "SL 552_001",
    "SL M117_003"
]

NUM_FINE = len(class_names)

# 2. HIERARCHY DEFINITIONS

# Fine → C2 (Brand × Type)
fine_to_c2 = {
    0: 0,  # GG0038ON_011 → Brand 1 Optical
    1: 1,  # GG0062S_003  → Brand 1 Sun
    2: 0,  # GG0131O_001  → Brand 1 Optical
    3: 2,  # SL 124_001   → SL Optical
    4: 3,  # SL 552_001   → SL Sun
    5: 2   # SL M117_003  → SL Optical
}

# C2 → C1 (Brand)
c2_to_c1 = {
    0: 0,  # Brand 1 Optical → Brand 1
    1: 0,  # Brand 1 Sun     → Brand 1
    2: 1,  # SL Optical    → Brand 2
    3: 1   # SL Sun        → Brand 2
}

NUM_C2 = 4
NUM_C1 = 2


# 3. LOAD DATASETS (FLAT LABELS)

train_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)


# 4. MAP FLAT → HIERARCHICAL LABELS

def map_to_hierarchy(images, fine_labels):
    fine_ids = tf.argmax(fine_labels, axis=1)

    c2_ids = tf.gather(
        tf.constant([fine_to_c2[i] for i in range(NUM_FINE)]),
        fine_ids
    )

    c1_ids = tf.gather(
        tf.constant([c2_to_c1[i] for i in range(NUM_C2)]),
        c2_ids
    )

    return images, {
        "c1_output": tf.one_hot(c1_ids, depth=NUM_C1),
        "c2_output": tf.one_hot(c2_ids, depth=NUM_C2),
        "fine_output": tf.one_hot(fine_ids, depth=NUM_FINE),
    }

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)


# 5. HCNN MODEL DEFINITION (VGG-STYLE BACKBONE)

inputs = layers.Input(shape=(img_height, img_width, 3))

# Normalize
x = layers.Rescaling(1.0 / 255)(inputs)

# ---- VGG-style feature extractor ----
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)
shared_features = layers.Dense(512, activation="relu")(x)

# ---- Hierarchical heads ----
c1_output = layers.Dense(NUM_C1, activation="softmax", name="c1_output")(shared_features)
c2_output = layers.Dense(NUM_C2, activation="softmax", name="c2_output")(shared_features)
fine_output = layers.Dense(NUM_FINE, activation="softmax", name="fine_output")(shared_features)

model = models.Model(
    inputs=inputs,
    outputs=[c1_output, c2_output, fine_output],
    name="Hierarchical_CNN"
)

model.summary()


# 6. COMPILE MODEL

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss={
        "c1_output": "categorical_crossentropy",
        "c2_output": "categorical_crossentropy",
        "fine_output": "categorical_crossentropy",
    },
    loss_weights={
        "c1_output": 0.2,
        "c2_output": 0.3,
        "fine_output": 0.5,
    },
    metrics={
        "c1_output": "accuracy",
        "c2_output": "accuracy",
        "fine_output": "accuracy",
    }
)


# 7. LEARNING RATE SCHEDULE (30-EPOCH SAFE)

def lr_scheduler(epoch, lr):
    if epoch < 20:
        return 1e-3
    return 2e-4


# 8. CALLBACKS

callbacks = [
    LearningRateScheduler(lr_scheduler, verbose=1),
    EarlyStopping(
        monitor="val_fine_output_loss",
        patience=5,
        restore_best_weights=True,
        mode='min' # Added to fix the error
    ),
    TensorBoard(log_dir="./logs_30epoch")
]


# 9. TRAIN (30 EPOCHS ONLY)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("✅ HCNN training complete (30 epochs)")


Found 420 files belonging to 6 classes.
Using 336 files for training.
Found 420 files belonging to 6 classes.
Using 84 files for validation.


Model: "Hierarchical_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ rescaling[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ max_pooling2d_2[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    131,584 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c1_output (Dense)   │ (None, 2)         │      1,026 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c2_output (Dense)   │ (None, 4)         │      2,052 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fine_output (Dense) │ (None, 6)         │      3,078 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,283,148 (4.89 MB)

 Trainable params: 1,283,148 (4.89 MB)

 Non-trainable params: 0 (0.00 B)


Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 101s 8s/step - c1_output_accuracy: 0.4546 - c1_output_loss: 0.6969 - c2_output_accuracy: 0.2869 - c2_output_loss: 1.3576 - fine_output_accuracy: 0.1872 - fine_output_loss: 1.7971 - loss: 1.4455 - val_c1_output_accuracy: 0.5238 - val_c1_output_loss: 0.6931 - val_c2_output_accuracy: 0.3333 - val_c2_output_loss: 1.3363 - val_fine_output_accuracy: 0.1310 - val_fine_output_loss: 1.8030 - val_loss: 1.4402 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 86s 8s/step - c1_output_accuracy: 0.5427 - c1_output_loss: 0.6935 - c2_output_accuracy: 0.2991 - c2_output_loss: 1.3533 - fine_output_accuracy: 0.1807 - fine_output_loss: 1.7970 - loss: 1.4435 - val_c1_output_accuracy: 0.5238 - val_c1_output_loss: 0.6911 - val_c2_output_accuracy: 0.3333 - val_c2_output_loss: 1.3337 - val_fine_output_accuracy: 0.1786 - val_fine_output

In [ ]:
# 10. evaluate the model

print("\n--- Training Summary ---")
print(f"Total epochs trained: {len(history.history['loss'])}")

print("\nFinal Training Metrics:")
print(f"  Overall Loss: {history.history['loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['fine_output_accuracy'][-1]:.4f}")

print("\nFinal Validation Metrics:")
print(f"  Overall Loss: {history.history['val_loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['val_c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['val_c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['val_fine_output_accuracy'][-1]:.4f}")

# Find the epoch with the best validation fine output loss
best_epoch = history.history['val_fine_output_loss'].index(min(history.history['val_fine_output_loss']))
print(f"\nBest Validation Fine Output Loss occurred at epoch {best_epoch + 1}:")
print(f"  Val Fine Output Loss: {history.history['val_fine_output_loss'][best_epoch]:.4f}")
print(f"  Val Fine Output Accuracy: {history.history['val_fine_output_accuracy'][best_epoch]:.4f}")



--- Training Summary ---
Total epochs trained: 7

Final Training Metrics:
  Overall Loss: 1.4350
  C1 Accuracy: 0.4524
  C2 Accuracy: 0.3333
  Fine Accuracy: 0.1756

Final Validation Metrics:
  Overall Loss: 1.4374
  C1 Accuracy: 0.4762
  C2 Accuracy: 0.3333
  Fine Accuracy: 0.1310

Best Validation Fine Output Loss occurred at epoch 2:
  Val Fine Output Loss: 1.7905
  Val Fine Output Accuracy: 0.1786


In [ ]:
##START VGG16 HCNN MODEL

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping


# 1. CONFIGURATION

image_dir = "/content/sample_data"   # CHANGE if needed
img_height = 224
img_width = 224
batch_size = 32
EPOCHS = 30

class_names = [
    "GG0038ON_011",
    "GG0062S_003",
    "GG0131O_001",
    "SL 124_001",
    "SL 552_001",
    "SL M117_003"
]

NUM_FINE = len(class_names)



# 2. HIERARCHY DEFINITIONS

# Fine → C2 (Brand × Type)
fine_to_c2 = {
    0: 0,  # GG0038ON_011 → Brand 1 Optical
    1: 1,  # GG0062S_003  → Brand 1 Sun
    2: 0,  # GG0131O_001  → Brand 1 Optical
    3: 2,  # SL 124_001   → SL Optical
    4: 3,  # SL 552_001   → SL Sun
    5: 2   # SL M117_003  → SL Optical
}

# C2 → C1 (Brand)
c2_to_c1 = {
    0: 0,  # Brand 1 Optical → Brand 1
    1: 0,  # Brand 1 Sun     → Brand 1
    2: 1,  # SL Optical    → Brand 2
    3: 1   # SL Sun        → Brand 2
}

NUM_C2 = 4
NUM_C1 = 2


# 3. LOAD DATASETS (FLAT LABELS)

train_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)


# 4. MAP FLAT → HIERARCHICAL LABELS

def map_to_hierarchy(images, fine_labels):
    fine_ids = tf.argmax(fine_labels, axis=1)

    c2_ids = tf.gather(
        tf.constant([fine_to_c2[i] for i in range(NUM_FINE)]),
        fine_ids
    )

    c1_ids = tf.gather(
        tf.constant([c2_to_c1[i] for i in range(NUM_C2)]),
        c2_ids
    )

    return images, {
        "c1_output": tf.one_hot(c1_ids, depth=NUM_C1),
        "c2_output": tf.one_hot(c2_ids, depth=NUM_C2),
        "fine_output": tf.one_hot(fine_ids, depth=NUM_FINE),
    }

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)


# 5. VGG16 BACKBONE (TRANSFER LEARNING)

base_model = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(img_height, img_width, 3)
)

# Freeze backbone
for layer in base_model.layers:
    layer.trainable = False

inputs = layers.Input(shape=(img_height, img_width, 3))
x = tf.keras.applications.vgg16.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)


# 6. HIERARCHICAL OUTPUT HEADS

c1_output = layers.Dense(NUM_C1, activation="softmax", name="c1_output")(x)
c2_output = layers.Dense(NUM_C2, activation="softmax", name="c2_output")(x)
fine_output = layers.Dense(NUM_FINE, activation="softmax", name="fine_output")(x)

model = models.Model(
    inputs=inputs,
    outputs=[c1_output, c2_output, fine_output],
    name="HCNN_VGG16_Transfer"
)

model.summary()


# 7. COMPILE (FINE-LEVEL PRIORITIZED)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "c1_output": "categorical_crossentropy",
        "c2_output": "categorical_crossentropy",
        "fine_output": "categorical_crossentropy"
    },
    loss_weights={
        "c1_output": 0.1,
        "c2_output": 0.2,
        "fine_output": 0.7
    },
    metrics={
        "c1_output": "accuracy",
        "c2_output": "accuracy",
        "fine_output": "accuracy"
    }
)


# 8. LR SCHEDULE + EARLY STOPPING

def lr_scheduler(epoch, lr):
    if epoch < 20:
        return 1e-3
    return 2e-4

EarlyStopping(
    monitor="val_fine_output_loss",
    mode="min",
    patience=6,
    restore_best_weights=True
)

callbacks = [
    tf.keras.callbacks.LearningRateScheduler(lr_scheduler, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_fine_output_loss",
        mode="min",          # ← REQUIRED FIX
        patience=6,
        restore_best_weights=True
    )
]


# 9. TRAIN (30 EPOCHS MAX)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("✅ HCNN VGG16 transfer learning training complete")


Found 420 files belonging to 6 classes.
Using 336 files for training.
Found 420 files belonging to 6 classes.
Using 84 files for validation.


Model: "HCNN_VGG16_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer_2[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer_2[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 7, 7, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    262,656 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c1_output (Dense)   │ (None, 2)         │      1,026 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c2_output (Dense)   │ (None, 4)         │      2,052 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fine_output (Dense) │ (None, 6)         │      3,078 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,983,500 (57.16 MB)

 Trainable params: 268,812 (1.03 MB)

 Non-trainable params: 14,714,688 (56.13 MB)


Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 73s 6s/step - c1_output_accuracy: 0.5086 - c1_output_loss: 2.4412 - c2_output_accuracy: 0.2888 - c2_output_loss: 5.2235 - fine_output_accuracy: 0.1817 - fine_output_loss: 6.4463 - loss: 5.7633 - val_c1_output_accuracy: 0.4762 - val_c1_output_loss: 1.0740 - val_c2_output_accuracy: 0.4405 - val_c2_output_loss: 1.9118 - val_fine_output_accuracy: 0.4762 - val_fine_output_loss: 2.4053 - val_loss: 2.1576 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 54s 5s/step - c1_output_accuracy: 0.5480 - c1_output_loss: 1.5181 - c2_output_accuracy: 0.3867 - c2_output_loss: 2.8399 - fine_output_accuracy: 0.4540 - fine_output_loss: 2.9071 - loss: 2.7278 - val_c1_output_accuracy: 0.5357 - val_c1_output_loss: 0.8616 - val_c2_output_accuracy: 0.5000 - val_c2_output_loss: 1.4265 - val_fine_output_accuracy: 0.6548 - val_fine_output_

In [ ]:
# 10. Evaluate the model

print("\n--- Training Summary ---")
print(f"Total epochs trained: {len(history.history['loss'])}")

print("\nFinal Training Metrics:")
print(f"  Overall Loss: {history.history['loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['fine_output_accuracy'][-1]:.4f}")

print("\nFinal Validation Metrics:")
print(f"  Overall Loss: {history.history['val_loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['val_c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['val_c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['val_fine_output_accuracy'][-1]:.4f}")

# Find the epoch with the best validation fine output loss
best_epoch = history.history['val_fine_output_loss'].index(min(history.history['val_fine_output_loss']))
print(f"\nBest Validation Fine Output Loss occurred at epoch {best_epoch + 1}:")
print(f"  Val Fine Output Loss: {history.history['val_fine_output_loss'][best_epoch]:.4f}")
print(f"  Val Fine Output Accuracy: {history.history['val_fine_output_accuracy'][best_epoch]:.4f}")


--- Training Summary ---
Total epochs trained: 25

Final Training Metrics:
  Overall Loss: 0.0454
  C1 Accuracy: 0.9851
  C2 Accuracy: 0.9821
  Fine Accuracy: 0.9970

Final Validation Metrics:
  Overall Loss: 0.4132
  C1 Accuracy: 0.8571
  C2 Accuracy: 0.8571
  Fine Accuracy: 0.8452

Best Validation Fine Output Loss occurred at epoch 19:
  Val Fine Output Loss: 0.4259
  Val Fine Output Accuracy: 0.8333


In [ ]:
##START VGG19 HCNN MODEL

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping


# 1. CONFIGURATION

image_dir = "/content/sample_data"   # CHANGE if needed
img_height = 224
img_width = 224
batch_size = 32
EPOCHS = 30

class_names = [
    "GG0038ON_011",
    "GG0062S_003",
    "GG0131O_001",
    "SL 124_001",
    "SL 552_001",
    "SL M117_003"
]

NUM_FINE = len(class_names)


# 2. HIERARCHY DEFINITIONS

# Fine → C2 (Brand × Type)
fine_to_c2 = {
    0: 0,  # GG0038ON_011 → Brand 1 Optical
    1: 1,  # GG0062S_003  → Brand 1 Sun
    2: 0,  # GG0131O_001  → Brand 1 Optical
    3: 2,  # SL 124_001   → SL Optical
    4: 3,  # SL 552_001   → SL Sun
    5: 2   # SL M117_003  → SL Optical
}

# C2 → C1 (Brand)
c2_to_c1 = {
    0: 0,  # Brand 1 Optical → Brand 1
    1: 0,  # Brand 1 Sun     → Brand 1
    2: 1,  # SL Optical    → Brand 2
    3: 1   # SL Sun        → Brand 2
}

NUM_C2 = 4
NUM_C1 = 2


# 3. LOAD DATASETS (FLAT LABELS)

train_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    class_names=class_names,
    label_mode="categorical"
)


# 4. MAP FLAT → HIERARCHICAL LABELS

def map_to_hierarchy(images, fine_labels):
    fine_ids = tf.argmax(fine_labels, axis=1)

    c2_ids = tf.gather(
        tf.constant([fine_to_c2[i] for i in range(NUM_FINE)]),
        fine_ids
    )

    c1_ids = tf.gather(
        tf.constant([c2_to_c1[i] for i in range(NUM_C2)]),
        c2_ids
    )

    return images, {
        "c1_output": tf.one_hot(c1_ids, depth=NUM_C1),
        "c2_output": tf.one_hot(c2_ids, depth=NUM_C2),
        "fine_output": tf.one_hot(fine_ids, depth=NUM_FINE),
    }

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(map_to_hierarchy, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)


# 5. VGG16 BACKBONE (TRANSFER LEARNING)

base_model = VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(img_height, img_width, 3)
)

# Freeze backbone
for layer in base_model.layers:
    layer.trainable = False

inputs = layers.Input(shape=(img_height, img_width, 3))
x = tf.keras.applications.vgg19.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)


# 6. HIERARCHICAL OUTPUT HEADS

c1_output = layers.Dense(NUM_C1, activation="softmax", name="c1_output")(x)
c2_output = layers.Dense(NUM_C2, activation="softmax", name="c2_output")(x)
fine_output = layers.Dense(NUM_FINE, activation="softmax", name="fine_output")(x)

model = models.Model(
    inputs=inputs,
    outputs=[c1_output, c2_output, fine_output],
    name="HCNN_VGG19_Transfer"
)

model.summary()


# 7. COMPILE (FINE-LEVEL PRIORITIZED)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "c1_output": "categorical_crossentropy",
        "c2_output": "categorical_crossentropy",
        "fine_output": "categorical_crossentropy"
    },
    loss_weights={
        "c1_output": 0.1,
        "c2_output": 0.2,
        "fine_output": 0.7
    },
    metrics={
        "c1_output": "accuracy",
        "c2_output": "accuracy",
        "fine_output": "accuracy"
    }
)


# 8. LR SCHEDULE + EARLY STOPPING

def lr_scheduler(epoch, lr):
    if epoch < 20:
        return 1e-3
    return 2e-4

EarlyStopping(
    monitor="val_fine_output_loss",
    mode="min",
    patience=6,
    restore_best_weights=True
)

callbacks = [
    tf.keras.callbacks.LearningRateScheduler(lr_scheduler, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_fine_output_loss",
        mode="min",          # ← REQUIRED FIX
        patience=6,
        restore_best_weights=True
    )
]


# 9. TRAIN (30 EPOCHS MAX)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("✅ HCNN VGG19 transfer learning training complete")


Found 420 files belonging to 6 classes.
Using 336 files for training.
Found 420 files belonging to 6 classes.
Using 84 files for validation.


Model: "HCNN_VGG19_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg19 (Functional)  │ (None, 7, 7, 512) │ 20,024,384 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg19[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    262,656 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c1_output (Dense)   │ (None, 2)         │      1,026 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c2_output (Dense)   │ (None, 4)         │      2,052 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fine_output (Dense) │ (None, 6)         │      3,078 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,293,196 (77.41 MB)

 Trainable params: 268,812 (1.03 MB)

 Non-trainable params: 20,024,384 (76.39 MB)


Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 78s 6s/step - c1_output_accuracy: 0.5312 - c1_output_loss: 3.2141 - c2_output_accuracy: 0.3056 - c2_output_loss: 4.6732 - fine_output_accuracy: 0.1644 - fine_output_loss: 7.7263 - loss: 6.6543 - val_c1_output_accuracy: 0.5119 - val_c1_output_loss: 1.0291 - val_c2_output_accuracy: 0.5595 - val_c2_output_loss: 1.3429 - val_fine_output_accuracy: 0.3929 - val_fine_output_loss: 3.1996 - val_loss: 2.5624 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 67s 6s/step - c1_output_accuracy: 0.5605 - c1_output_loss: 1.8983 - c2_output_accuracy: 0.4085 - c2_output_loss: 2.9401 - fine_output_accuracy: 0.4170 - fine_output_loss: 3.3715 - loss: 3.1311 - val_c1_output_accuracy: 0.5357 - val_c1_output_loss: 1.0548 - val_c2_output_accuracy: 0.5238 - val_c2_output_loss: 1.0812 - val_fine_output_accuracy: 0.5714 - val_fine_output_

In [ ]:
# 10. Evaluate the model

print("\n--- Training Summary ---")
print(f"Total epochs trained: {len(history.history['loss'])}")

print("\nFinal Training Metrics:")
print(f"  Overall Loss: {history.history['loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['fine_output_accuracy'][-1]:.4f}")

print("\nFinal Validation Metrics:")
print(f"  Overall Loss: {history.history['val_loss'][-1]:.4f}")
print(f"  C1 Accuracy: {history.history['val_c1_output_accuracy'][-1]:.4f}")
print(f"  C2 Accuracy: {history.history['val_c2_output_accuracy'][-1]:.4f}")
print(f"  Fine Accuracy: {history.history['val_fine_output_accuracy'][-1]:.4f}")

# Find the epoch with the best validation fine output loss
best_epoch = history.history['val_fine_output_loss'].index(min(history.history['val_fine_output_loss']))
print(f"\nBest Validation Fine Output Loss occurred at epoch {best_epoch + 1}:")
print(f"  Val Fine Output Loss: {history.history['val_fine_output_loss'][best_epoch]:.4f}")
print(f"  Val Fine Output Accuracy: {history.history['val_fine_output_accuracy'][best_epoch]:.4f}")


--- Training Summary ---
Total epochs trained: 19

Final Training Metrics:
  Overall Loss: 0.0933
  C1 Accuracy: 0.9315
  C2 Accuracy: 0.9524
  Fine Accuracy: 0.9702

Final Validation Metrics:
  Overall Loss: 0.4688
  C1 Accuracy: 0.8452
  C2 Accuracy: 0.8214
  Fine Accuracy: 0.8452

Best Validation Fine Output Loss occurred at epoch 13:
  Val Fine Output Loss: 0.5164
  Val Fine Output Accuracy: 0.7976
